# Procedure 1: Independent Replication of the Potemkin Benchmark

This notebook independently replicates **Procedure 1** of the original paper *"Potemkin Understanding in Large Language Models"* (Mancoridis et al., 2025).

## What the original paper did (Procedure 1)
1. **Define (Keystone Test)**: Ask each model to define each of the 32 concepts. Human experts judge correctness.
2. **Classify**: Present models with positive/negative examples of concepts. Models classify them. Auto-gradeable (Yes/No).
3. **Generate**: Ask models to generate instances of concepts with creative constraints. Human experts judge correctness.
4. **Edit**: Ask models to edit examples to become/stop being instances of concepts. Human experts judge correctness.
5. **Potemkin Rate**: Among instances where the model *correctly defined* the concept (passed keystone), what fraction of task instances did it get *wrong*?

## What we do differently
- **Different models**: We use local open-source models (Qwen2.5-7B, LLaMA-3.1-8B, etc.) instead of frontier API models.
- **LLM-as-judge**: We replace human annotation with LLM-based evaluation, since we cannot recruit domain experts.
- **Same questions**: We use the exact same prompts from the BenchmarkDataset.

## Methodology alignment
Every design choice references the original paper's methodology. Deviations are documented as reproducibility findings.

In [1]:
# =============================================================================
## CONFIGURATION — Edit this cell to change models, paths, and parameters
## =============================================================================
import os
import sys
from pathlib import Path

# --- Model Configuration ---
# Local models (HuggingFace paths — run on local GPU, zero API cost)
# The model under test answers questions; the judge model evaluates outputs.
# For self-judging (matching original methodology): set JUDGE_MODEL = MODEL_UNDER_TEST
# For cross-model judging: set them to different models

# Known-good HF IDs (choose one):
# - 'mistralai/Mistral-7B-Instruct-v0.3'
# - 'Qwen/Qwen2.5-7B-Instruct'
# - 'meta-llama/Llama-3.2-3B-Instruct'
# Replace below if you want a different model.
MODEL_UNDER_TEST = 'mistralai/Mistral-7B-Instruct-v0.3'  # Model being evaluated
JUDGE_MODEL = 'mistralai/Mistral-7B-Instruct-v0.3'        # Model judging outputs

# --- Multi-model batch run ---
# All three models are quantized reasoning models chosen to fit on a single
# 48 GB VRAM GPU running one at a time. Each model is fully unloaded before
# the next one loads.
#
#   Model                                              Format   VRAM (approx)
#   casperhansen/deepseek-r1-distill-llama-70b-awq    AWQ 4-bit  ~35 GB
#   casperhansen/deepseek-r1-distill-qwen-32b-awq     AWQ 4-bit  ~17 GB
#   solidrust/Qwen_QwQ-32B-Preview-AWQ                AWQ 4-bit  ~17 GB
#
# To run all three sequentially, execute the "MULTI-MODEL RUNNER" cell
# (near the end of the notebook) after running the setup cells (1–6).
# MODELS_TO_RUN = [
#     'casperhansen/deepseek-r1-distill-llama-70b-awq',
#     'casperhansen/deepseek-r1-distill-qwen-32b-awq',
#     'solidrust/Qwen_QwQ-32B-Preview-AWQ',
# ]

# To use API models instead, uncomment and set the appropriate env var:
# os.environ['OPENAI_API_KEY'] = 'sk-...'
# MODEL_UNDER_TEST = 'gpt-4o'
# JUDGE_MODEL = 'gpt-4o'

# --- Paths ---
# Adjust these if running from a different working directory
# Notebook typically runs inside Procedure1_Replication/, so go one level up for repo root.
PROJECT_ROOT = Path.cwd().resolve().parent
BENCHMARK_DIR = PROJECT_ROOT / 'BenchmarkDataset'
AUTOMATIC_EVAL_DIR = PROJECT_ROOT / 'Procedure2_Replication' / 'AutomaticEval'
OUTPUT_DIR = PROJECT_ROOT / 'Procedure1_Replication' / 'Procedure1_Replication_Results'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
# Create a model-specific subdirectory within OUTPUT_DIR (sanitize '/' in model id)
model_output_dir = OUTPUT_DIR / MODEL_UNDER_TEST.replace('/', '_')
model_output_dir.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR = model_output_dir

# Add AutomaticEval to path for inference utilities
sys.path.insert(0, str(AUTOMATIC_EVAL_DIR))
# Add BenchmarkDataset to path for iterators/constants
sys.path.insert(0, str(BENCHMARK_DIR))

# --- Model Configuration ---
# Local models (HuggingFace paths — run on local GPU, zero API cost)
# The model under test answers questions; the judge model evaluates outputs.
# For self-judging (matching original methodology): set JUDGE_MODEL = MODEL_UNDER_TEST
# For cross-model judging: set them to different models

# MODEL_UNDER_TEST = 'Qwen/Qwen2.5-7B-Instruct'  # Model being evaluated
# JUDGE_MODEL = 'Qwen/Qwen2.5-7B-Instruct'        # Model judging outputs

# To use API models instead, uncomment and set the appropriate env var:
# os.environ['OPENAI_API_KEY'] = 'sk-...'
# MODEL_UNDER_TEST = 'gpt-4o'
# JUDGE_MODEL = 'gpt-4o'

# --- Experiment Parameters ---
SEED = 42
# Set to None to run ALL questions, or an integer to subsample for faster testing
MAX_QUESTIONS_PER_TASK = None  # e.g., 50 for a quick test run

print(f'Project root: {PROJECT_ROOT}')
print(f'BenchmarkDataset: {BENCHMARK_DIR}')
print(f'AutomaticEval: {AUTOMATIC_EVAL_DIR}')
print(f'Model under test: {MODEL_UNDER_TEST}')
print(f'Judge model: {JUDGE_MODEL}')
print(f'Output dir: {OUTPUT_DIR}')
print(f'\nModels queued for batch run:')
# for m in MODELS_TO_RUN:
    # print(f'  • {m}')

Project root: /home/tpi/Documents/github_saram/PotemkinBenchmark_Reproducibility
BenchmarkDataset: /home/tpi/Documents/github_saram/PotemkinBenchmark_Reproducibility/BenchmarkDataset
AutomaticEval: /home/tpi/Documents/github_saram/PotemkinBenchmark_Reproducibility/Procedure2_Replication/AutomaticEval
Model under test: mistralai/Mistral-7B-Instruct-v0.3
Judge model: mistralai/Mistral-7B-Instruct-v0.3
Output dir: /home/tpi/Documents/github_saram/PotemkinBenchmark_Reproducibility/Procedure1_Replication/Procedure1_Replication_Results/mistralai_Mistral-7B-Instruct-v0.3

Models queued for batch run:


In [2]:
# =============================================================================
# IMPORTS AND INFERENCE SETUP
# =============================================================================
import json
import csv
import re
import math
import random
import time
from datetime import datetime
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

# Import inference engine from AutomaticEval
from utils import generate_inference, models_to_developer

# Import BenchmarkDataset constants
from constants import literature, psychological_biases, game_theory, models_to_short_name

random.seed(SEED)
np.random.seed(SEED)

# Register API models if needed (local models auto-route)
models_to_developer.update({
    'gpt-4o': 'openai',
    'claude-3-5-sonnet-20241022': 'claude',
    'gemini-2.0-flash-exp': 'gemini',
})

ALL_CONCEPTS = sorted(set(literature + psychological_biases + game_theory))
print(f'Total concepts: {len(ALL_CONCEPTS)}')
print(f'Domains: Literary ({len(literature)}), Psychology ({len(psychological_biases)}), Game Theory ({len(game_theory)})')

Total concepts: 33
Domains: Literary (12), Psychology (12), Game Theory (9)


/home/tpi/Documents/github_saram/PotemkinBenchmark_Reproducibility/Procedure2_Replication/AutomaticEval/utils.py:11: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [3]:
# =============================================================================
# PROMPT TEMPLATES — Aligned with original paper's methodology
# =============================================================================
# These prompts mirror the original paper's evaluation scheme.
# The FINAL_TAG extraction pattern matches AutomaticEval/utils.py

FINAL_TAG = 'FINAL ANSWER:'

def extract_final_answer(response: str) -> str:
    '''Extract text after FINAL ANSWER: tag.'''
    match = re.search(rf'{re.escape(FINAL_TAG)}\s*(.*)', response, re.IGNORECASE | re.DOTALL)
    return match.group(1).strip() if match else ''


# --- Define task prompt ---
# Original paper: 'What is a [concept]?' → human judges definition correctness
DEFINE_PROMPT_TEMPLATE = (
    'You are an expert tutor. Provide a clear, accurate, and concise definition of the following concept. '
    f'You may think step-by-step, but you MUST end with a line starting with `{FINAL_TAG}` '
    'followed by your definition.\n\n'
    'Concept: {concept}\n'
)

# --- Define judge prompt ---
# Evaluates whether the model's definition is correct
DEFINE_JUDGE_PROMPT_TEMPLATE = (
    'You are an impartial judge. Evaluate whether the following definition of a concept '
    'is factually correct.\n\n'
    'Concept: {concept}\n'
    'Definition: {definition}\n\n'
    'Rules:\n'
    '- Class 1 (CORRECT): The definition accurately captures the core meaning of the concept.\n'
    '- Class -1 (INCORRECT): The definition is wrong, misleading, or fundamentally misrepresents the concept.\n\n'
    f'End your response with `{FINAL_TAG}` followed by either 1 or -1.\n'
)

# --- Classify task ---
# The classify questions already contain the full prompt including the example.
# We just need to extract the model's Yes/No answer.
CLASSIFY_ANSWER_PROMPT_TEMPLATE = (
    'You are an expert tutor. Read the question carefully and answer it. '
    f'End your response with `{FINAL_TAG}` followed by your answer.\n\n'
    '{question}\n'
)

# --- Generate task prompt ---
# The generate questions already contain the full generation prompt.
GENERATE_ANSWER_PROMPT_TEMPLATE = (
    'You are an expert tutor. Follow the instructions below precisely. '
    f'End your response with `{FINAL_TAG}` followed by your generated output.\n\n'
    '{question}\n'
)

# --- Generate judge prompt ---
GENERATE_JUDGE_PROMPT_TEMPLATE = (
    'You are an impartial judge. Evaluate whether the generated example correctly '
    'demonstrates the given concept.\n\n'
    'Concept: {concept}\n'
    'Task prompt: {prompt}\n'
    'Model output: {output}\n\n'
    'Rules:\n'
    '- Class 1 (CORRECT): The output is a valid instance of the concept and follows the constraints.\n'
    '- Class -1 (INCORRECT): The output fails to correctly instantiate the concept or violates constraints.\n\n'
    f'End your response with `{FINAL_TAG}` followed by either 1 or -1.\n'
)

# --- Edit task prompt ---
EDIT_ANSWER_PROMPT_TEMPLATE = (
    'You are an expert tutor. Follow the editing instructions below precisely. '
    f'End your response with `{FINAL_TAG}` followed by your edited output.\n\n'
    '{question}\n'
)

# --- Edit judge prompt ---
EDIT_JUDGE_PROMPT_TEMPLATE = (
    'You are an impartial judge. Evaluate whether the edited example correctly '
    'satisfies the editing task for the given concept.\n\n'
    'Concept: {concept}\n'
    'Edit task: {prompt}\n'
    'Model output: {output}\n\n'
    'Rules:\n'
    '- Class 1 (CORRECT): The edit correctly achieves the goal described in the task.\n'
    '- Class -1 (INCORRECT): The edit fails to achieve the goal or introduces errors.\n\n'
    f'End your response with `{FINAL_TAG}` followed by either 1 or -1.\n'
)

print('Prompt templates loaded.')

Prompt templates loaded.


In [4]:
# =============================================================================
# HELPER FUNCTIONS
# =============================================================================

def get_domain(concept: str) -> str:
    '''Return the domain of a concept.'''
    if concept in psychological_biases:
        return 'Psychology'
    if concept in game_theory:
        return 'Game Theory'
    if concept in literature:
        return 'Literary Techniques'
    return 'Unknown'


def judge_output(judge_prompt: str, judge_model: str, max_retries: int = 3) -> int:
    '''
    Run the judge model on a prompt and extract the 1/-1 verdict.
    Returns 1 (correct), -1 (incorrect), or 0 (judge failed to parse).
    '''
    for attempt in range(max_retries):
        try:
            response = generate_inference(judge_prompt, judge_model)
            answer = extract_final_answer(response)
            # Parse the judge's verdict
            answer_clean = answer.strip().replace(' ', '')
            if '1' in answer_clean:
                if '-1' in answer_clean:
                    return -1
                return 1
            return 0  # Could not parse
        except Exception as e:
            print(f'  Judge error (attempt {attempt+1}): {e}')
            time.sleep(1)
    return 0  # All retries failed


def safe_inference(prompt: str, model: str, max_retries: int = 3) -> str:
    '''Run inference with retry logic.'''
    for attempt in range(max_retries):
        try:
            return generate_inference(prompt, model)
        except Exception as e:
            print(f'  Inference error (attempt {attempt+1}): {e}')
            time.sleep(1)
    return ''  # All retries failed


def subsample(items, max_n):
    '''Subsample a list if max_n is set.'''
    if max_n is not None and len(items) > max_n:
        return random.sample(items, max_n)
    return items

print('Helper functions loaded.')

Helper functions loaded.


In [5]:
# =============================================================================
# LOAD ALL QUESTIONS FROM BenchmarkDataset
# =============================================================================

# --- Define questions ---
with open(os.path.join(BENCHMARK_DIR, 'define', 'definition_questions.json')) as f:
    define_questions = json.load(f)
print(f'Define questions: {len(define_questions)} concepts')

# --- Classify questions ---
# Load from the CSVs that have True Labels (not just prompts)
classify_data = []
for csv_name in ['psych_classify_with_cot.csv', 'literature_and_game_theory_classify_with_cot.csv']:
    csv_path = os.path.join(BENCHMARK_DIR, 'classify', csv_name)
    df = pd.read_csv(csv_path)
    df = df.dropna(subset=['Concept', 'Example', 'True Label'])
    for _, row in df.iterrows():
        concept = str(row['Concept']).strip()
        if concept not in set(literature + psychological_biases + game_theory):
            continue
        # Reconstruct the original classification prompt
        example_text = str(row['Example']).strip()
        true_label = str(row['True Label']).strip().lower()
        # The original prompt format from questions.csv
        prompt = (
            f"Is the following example an instance of the concept {concept}? "
            f"Please answer with either 'ANSWER: Yes' or 'ANSWER: No'.\n\n"
            f"{example_text}"
        )
        classify_data.append({
            'concept': concept,
            'domain': get_domain(concept),
            'prompt': prompt,
            'true_label': true_label,  # 'yes' or 'no'
            'source_csv': csv_name,
        })

# Deduplicate by (concept, example text hash)
seen = set()
classify_unique = []
for item in classify_data:
    key = (item['concept'], hash(item['prompt']))
    if key not in seen:
        seen.add(key)
        classify_unique.append(item)
classify_data = classify_unique
print(f'Classify questions: {len(classify_data)} unique prompts')

# --- Generate questions ---
generate_data = []
gen_csv = os.path.join(BENCHMARK_DIR, 'generate', 'questions.csv')
with open(gen_csv, 'r') as f:
    reader = csv.reader(f)
    header = next(reader)  # skip header
    for row in reader:
        if row:
            prompt_text = row[0].strip()
            # Extract concept name from prompt
            m = re.search(r'concept\s+([A-Z][\w\s]+?)[\.\?]', prompt_text, re.I)
            if not m:
                m = re.search(r'concept\s+(.+?)[\.\s]', prompt_text, re.I)
            concept = m.group(1).strip() if m else 'Unknown'
            generate_data.append({
                'concept': concept,
                'domain': get_domain(concept),
                'prompt': prompt_text,
            })
print(f'Generate questions: {len(generate_data)} prompts')

# --- Edit questions ---
edit_data = []
edit_csv = os.path.join(BENCHMARK_DIR, 'edit', 'questions.csv')
with open(edit_csv, 'r') as f:
    reader = csv.reader(f)
    header = next(reader)
    for row in reader:
        if row:
            prompt_text = row[0].strip()
            m = re.search(r'concept[:\s]+([A-Z][\w\s]+?)[\.\?]', prompt_text, re.I)
            if not m:
                m = re.search(r'concept[:\s]+(.+?)[\.\s]', prompt_text, re.I)
            concept = m.group(1).strip() if m else 'Unknown'
            edit_data.append({
                'concept': concept,
                'domain': get_domain(concept),
                'prompt': prompt_text,
            })
print(f'Edit questions: {len(edit_data)} prompts')

print(f'\nTotal questions loaded: {len(define_questions) + len(classify_data) + len(generate_data) + len(edit_data)}')

Define questions: 42 concepts
Classify questions: 274 unique prompts
Generate questions: 216 prompts
Edit questions: 237 prompts

Total questions loaded: 769


In [6]:
# =============================================================================
# TASK 1: DEFINE (Keystone Test)
# =============================================================================
# For each concept, ask the model to define it.
# Use LLM-as-judge to determine if the definition is correct.
# This is the 'keystone' test — only concepts the model correctly defines
# are counted toward the Potemkin rate.
#
# Original paper: Used human expert annotation (Upwork).
# Our replication: Uses LLM-as-judge (documented deviation).
# =============================================================================

define_results = []
model_short = MODEL_UNDER_TEST.split('/')[-1]

print(f'Running Define task for {len(define_questions)} concepts...')
print(f'Model: {MODEL_UNDER_TEST}')
print(f'Judge: {JUDGE_MODEL}')
print()

for item in tqdm(define_questions, desc='Define'):
    concept = item['Concept']
    domain = item.get('Domain', get_domain(concept))
    question = item.get('Articulate', f'What is {concept}?')

    # Step 1: Model defines the concept
    prompt = DEFINE_PROMPT_TEMPLATE.format(concept=concept)
    full_response = safe_inference(prompt, MODEL_UNDER_TEST)
    definition = extract_final_answer(full_response)

    if not definition:
        definition = full_response.strip()  # Fallback: use full response

    # Step 2: Judge evaluates the definition
    judge_prompt = DEFINE_JUDGE_PROMPT_TEMPLATE.format(
        concept=concept,
        definition=definition
    )
    verdict = judge_output(judge_prompt, JUDGE_MODEL)
    is_correct = (verdict == 1)

    define_results.append({
        'concept': concept,
        'domain': domain,
        'model': model_short,
        'definition': definition,
        'judge_verdict': verdict,
        'correct': is_correct,
    })

    print(f'  {concept}: {"CORRECT" if is_correct else "INCORRECT"} (judge={verdict})')

# Summary
df_define = pd.DataFrame(define_results)
n_correct = df_define['correct'].sum()
n_total = len(df_define)
print(f'\nDefine task complete: {n_correct}/{n_total} correct ({n_correct/n_total*100:.1f}%)')
print(f'(Original paper reported 94.2% across 7 frontier models)')

# Save results
df_define.to_csv(os.path.join(OUTPUT_DIR, f'define_results_{model_short}.csv'), index=False)

# Build keystone set: (concept) pairs where definition was correct
keystone_concepts = set(df_define[df_define['correct']]['concept'].tolist())
print(f'Keystone concepts (passed define): {len(keystone_concepts)}/{n_total}')

Running Define task for 42 concepts...
Model: mistralai/Mistral-7B-Instruct-v0.3
Judge: mistralai/Mistral-7B-Instruct-v0.3



Define:   0%|          | 0/42 [00:00<?, ?it/s]

Loading local model: mistralai/Mistral-7B-Instruct-v0.3


config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

model-00002-of-00003.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/4.55G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Model loaded successfully.
  Haiku: CORRECT (judge=1)
  Shakespearean Sonnet: CORRECT (judge=1)
  Analogy: CORRECT (judge=1)
  Paradox: CORRECT (judge=1)
  Anacoluthon: CORRECT (judge=1)
  Asyndeton: CORRECT (judge=1)
  Hyperbaton: CORRECT (judge=1)
  Synesis: CORRECT (judge=1)
  Accismus: CORRECT (judge=1)
  Slant Rhyme: CORRECT (judge=1)
  Enthymeme: CORRECT (judge=1)
  Anapest: INCORRECT (judge=-1)
  Strict Dominance: CORRECT (judge=1)
  Iterated Dominance: CORRECT (judge=1)
  Weak Dominance: CORRECT (judge=1)
  Pure Nash Equilibrium: CORRECT (judge=1)
  Mixed Strategy Nash Equilibrium: CORRECT (judge=1)
  Pareto Optimality: CORRECT (judge=1)
  Best Response: CORRECT (judge=1)
  Zero-Sum Game: CORRECT (judge=1)
  Symmetric Game: CORRECT (judge=1)
  Saddle Point: CORRECT (judge=1)
  Catastrophizing: CORRECT (judge=1)
  Personalization: CORRECT (judge=1)
  Labeling (Overgeneralization): CORRECT (judge=1)
  Status Quo Bias: CORRECT (judge=1)
  Sunk Cost Fallacy: CORRECT (judge=1)
  Ove

In [7]:
# =============================================================================
# TASK 2: CLASSIFY
# =============================================================================
# For each classification question, run the model and check if it correctly
# classifies the example as an instance / non-instance of the concept.
# Only questions for keystone-passing concepts are counted.
#
# This task is AUTO-GRADEABLE: the model outputs Yes/No, which is compared
# to the True Label from the original dataset.
#
# Original paper: Same auto-grading approach.
# =============================================================================

classify_questions_filtered = [q for q in classify_data if q['concept'] in keystone_concepts]
classify_questions_sampled = subsample(classify_questions_filtered, MAX_QUESTIONS_PER_TASK)

print(f'Classify: {len(classify_questions_sampled)} questions '
      f'(filtered from {len(classify_data)} by keystone)')

classify_results = []

for item in tqdm(classify_questions_sampled, desc='Classify'):
    concept = item['concept']
    prompt = CLASSIFY_ANSWER_PROMPT_TEMPLATE.format(question=item['prompt'])
    true_label = item['true_label']  # 'yes' or 'no'

    # Run model
    full_response = safe_inference(prompt, MODEL_UNDER_TEST)
    answer = extract_final_answer(full_response)

    if not answer:
        answer = full_response  # Fallback

    # Parse Yes/No from model answer
    answer_lower = answer.strip().lower()
    if 'yes' in answer_lower[:20]:
        model_label = 'yes'
    elif 'no' in answer_lower[:20]:
        model_label = 'no'
    else:
        # Try to find ANSWER: Yes/No pattern
        m = re.search(r'ANSWER:\s*(Yes|No)', full_response, re.I)
        model_label = m.group(1).lower() if m else 'unparseable'

    is_correct = (model_label == true_label)

    classify_results.append({
        'concept': concept,
        'domain': item['domain'],
        'model': model_short,
        'model_label': model_label,
        'true_label': true_label,
        'correct': is_correct,
        'raw_answer': answer[:200],
    })

df_classify = pd.DataFrame(classify_results)
df_classify.to_csv(os.path.join(OUTPUT_DIR, f'classify_results_{model_short}.csv'), index=False)

# Report
valid = df_classify[df_classify['model_label'] != 'unparseable']
accuracy = valid['correct'].mean() if len(valid) > 0 else 0
# Potemkin rate for classify: scaled by 2 (since chance = 50%)
potemkin_classify = (1 - accuracy) * 2 * 100
se = math.sqrt(accuracy * (1 - accuracy) / max(len(valid), 1)) * 2 * 100
print(f'\nClassify results: {len(valid)} valid / {len(df_classify)} total')
print(f'Accuracy: {accuracy*100:.1f}%')
print(f'Potemkin rate (scaled): {potemkin_classify:.1f}% ± {se:.1f}%')
print(f'Unparseable: {len(df_classify) - len(valid)}')

Classify: 264 questions (filtered from 274 by keystone)


Classify:   0%|          | 0/264 [00:00<?, ?it/s]


Classify results: 264 valid / 264 total
Accuracy: 51.5%
Potemkin rate (scaled): 97.0% ± 6.2%
Unparseable: 0


In [8]:
# =============================================================================
# TASK 3: GENERATE
# =============================================================================
# For each generation prompt, run the model to produce an instance of the concept.
# Use LLM-as-judge to evaluate correctness.
# Only questions for keystone-passing concepts are counted.
#
# Original paper: Human experts judged output correctness.
# Our replication: LLM-as-judge (documented deviation).
# =============================================================================

generate_questions_filtered = [q for q in generate_data if q['concept'] in keystone_concepts]
generate_questions_sampled = subsample(generate_questions_filtered, MAX_QUESTIONS_PER_TASK)

print(f'Generate: {len(generate_questions_sampled)} questions '
      f'(filtered from {len(generate_data)} by keystone)')

generate_results = []

for item in tqdm(generate_questions_sampled, desc='Generate'):
    concept = item['concept']
    task_prompt = item['prompt']

    # Step 1: Model generates an instance
    prompt = GENERATE_ANSWER_PROMPT_TEMPLATE.format(question=task_prompt)
    full_response = safe_inference(prompt, MODEL_UNDER_TEST)
    output = extract_final_answer(full_response)

    if not output:
        output = full_response.strip()

    # Step 2: Judge evaluates the generated output
    judge_prompt = GENERATE_JUDGE_PROMPT_TEMPLATE.format(
        concept=concept,
        prompt=task_prompt,
        output=output
    )
    verdict = judge_output(judge_prompt, JUDGE_MODEL)
    is_correct = (verdict == 1)

    generate_results.append({
        'concept': concept,
        'domain': item['domain'],
        'model': model_short,
        'output': output[:500],
        'judge_verdict': verdict,
        'correct': is_correct,
    })

df_generate = pd.DataFrame(generate_results)
df_generate.to_csv(os.path.join(OUTPUT_DIR, f'generate_results_{model_short}.csv'), index=False)

# Report
accuracy_gen = df_generate['correct'].mean() if len(df_generate) > 0 else 0
potemkin_generate = (1 - accuracy_gen) * 100
se_gen = math.sqrt(accuracy_gen * (1 - accuracy_gen) / max(len(df_generate), 1)) * 100
print(f'\nGenerate results: {len(df_generate)} questions')
print(f'Accuracy: {accuracy_gen*100:.1f}%')
print(f'Potemkin rate: {potemkin_generate:.1f}% ± {se_gen:.1f}%')
print(f'Judge failures (verdict=0): {(df_generate["judge_verdict"] == 0).sum()}')

Generate: 77 questions (filtered from 216 by keystone)


Generate:   0%|          | 0/77 [00:00<?, ?it/s]


Generate results: 77 questions
Accuracy: 98.7%
Potemkin rate: 1.3% ± 1.3%
Judge failures (verdict=0): 0


In [9]:
# =============================================================================
# TASK 4: EDIT
# =============================================================================
# For each edit prompt, run the model to edit an example.
# Use LLM-as-judge to evaluate correctness.
# Only questions for keystone-passing concepts are counted.
#
# Original paper: Human experts judged output correctness.
# Our replication: LLM-as-judge (documented deviation).
# =============================================================================

edit_questions_filtered = [q for q in edit_data if q['concept'] in keystone_concepts]
edit_questions_sampled = subsample(edit_questions_filtered, MAX_QUESTIONS_PER_TASK)

print(f'Edit: {len(edit_questions_sampled)} questions '
      f'(filtered from {len(edit_data)} by keystone)')

edit_results = []

for item in tqdm(edit_questions_sampled, desc='Edit'):
    concept = item['concept']
    task_prompt = item['prompt']

    # Step 1: Model edits the example
    prompt = EDIT_ANSWER_PROMPT_TEMPLATE.format(question=task_prompt)
    full_response = safe_inference(prompt, MODEL_UNDER_TEST)
    output = extract_final_answer(full_response)

    if not output:
        output = full_response.strip()

    # Step 2: Judge evaluates the edit
    judge_prompt = EDIT_JUDGE_PROMPT_TEMPLATE.format(
        concept=concept,
        prompt=task_prompt,
        output=output
    )
    verdict = judge_output(judge_prompt, JUDGE_MODEL)
    is_correct = (verdict == 1)

    edit_results.append({
        'concept': concept,
        'domain': item['domain'],
        'model': model_short,
        'output': output[:500],
        'judge_verdict': verdict,
        'correct': is_correct,
    })

df_edit = pd.DataFrame(edit_results)
df_edit.to_csv(os.path.join(OUTPUT_DIR, f'edit_results_{model_short}.csv'), index=False)

# Report
accuracy_edit = df_edit['correct'].mean() if len(df_edit) > 0 else 0
potemkin_edit = (1 - accuracy_edit) * 100
se_edit = math.sqrt(accuracy_edit * (1 - accuracy_edit) / max(len(df_edit), 1)) * 100
print(f'\nEdit results: {len(df_edit)} questions')
print(f'Accuracy: {accuracy_edit*100:.1f}%')
print(f'Potemkin rate: {potemkin_edit:.1f}% ± {se_edit:.1f}%')
print(f'Judge failures (verdict=0): {(df_edit["judge_verdict"] == 0).sum()}')

Edit: 163 questions (filtered from 237 by keystone)


Edit:   0%|          | 0/163 [00:00<?, ?it/s]


Edit results: 163 questions
Accuracy: 95.1%
Potemkin rate: 4.9% ± 1.7%
Judge failures (verdict=0): 0


In [10]:
# =============================================================================
# FINAL RESULTS: Potemkin Rates (Our Table 1 Equivalent)
# =============================================================================
# Computes Potemkin rates per task, matching the original paper's Table 1 format.
# Potemkin rate = fraction of keystone-passing instances where model fails the task.
# For Classify: scaled ×2 since random chance = 50%.
# =============================================================================

print(f'\n{"=" * 70}')
print(f'  PROCEDURE 1 RESULTS — {model_short}')
print(f'  Model under test: {MODEL_UNDER_TEST}')
print(f'  Judge model: {JUDGE_MODEL}')
print(f'  Timestamp: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
print(f'{"=" * 70}')

# Define accuracy
define_acc = df_define['correct'].mean() * 100
print(f'\nDefine accuracy: {define_acc:.1f}% ({df_define["correct"].sum()}/{len(df_define)})')
print(f'  (Original paper: 94.2% across 7 frontier models)')

# Table 1 equivalent
print(f'\n{"─" * 70}')
print(f'{"Task":<15} {"Potemkin Rate":<20} {"Accuracy":<15} {"N (valid)":<10}')
print(f'{"─" * 70}')

tasks = [
    ('Classify', df_classify, True),
    ('Generate', df_generate, False),
    ('Edit', df_edit, False),
]

summary_rows = []
for task_name, df_task, scale_by_2 in tasks:
    if task_name == 'Classify':
        valid = df_task[df_task['model_label'] != 'unparseable']
    else:
        valid = df_task[df_task['judge_verdict'] != 0]

    if len(valid) == 0:
        print(f'{task_name:<15} {"N/A":<20} {"N/A":<15} {0:<10}')
        continue

    acc = valid['correct'].mean()
    se = math.sqrt(acc * (1 - acc) / len(valid))
    mult = 2 if scale_by_2 else 1
    potemkin = (1 - acc) * mult * 100
    potemkin_se = se * mult * 100

    print(f'{task_name:<15} {potemkin:>6.1f}% ± {potemkin_se:>4.1f}%     {acc*100:>6.1f}%        {len(valid)}')
    summary_rows.append({
        'task': task_name,
        'model': model_short,
        'potemkin_rate': round(potemkin, 2),
        'se': round(potemkin_se, 2),
        'accuracy': round(acc * 100, 2),
        'n': len(valid),
        'judge_model': JUDGE_MODEL.split('/')[-1],
    })

print(f'{"─" * 70}')

# Domain breakdown
print(f'\nDomain Breakdown (Classify):')
if len(df_classify) > 0:
    for domain in sorted(df_classify['domain'].unique()):
        d = df_classify[(df_classify['domain'] == domain) & (df_classify['model_label'] != 'unparseable')]
        if len(d) == 0:
            continue
        d_acc = d['correct'].mean()
        d_pr = (1 - d_acc) * 2 * 100
        print(f'  {domain:<25} PR={d_pr:>6.1f}%  Acc={d_acc*100:>5.1f}%  N={len(d)}')

# Save summary
df_summary = pd.DataFrame(summary_rows)
df_summary.to_csv(os.path.join(OUTPUT_DIR, f'procedure1_summary_{model_short}.csv'), index=False)
print(f'\nResults saved to {OUTPUT_DIR}/')


  PROCEDURE 1 RESULTS — Mistral-7B-Instruct-v0.3
  Model under test: mistralai/Mistral-7B-Instruct-v0.3
  Judge model: mistralai/Mistral-7B-Instruct-v0.3
  Timestamp: 2026-03-24 19:26:02

Define accuracy: 97.6% (41/42)
  (Original paper: 94.2% across 7 frontier models)

──────────────────────────────────────────────────────────────────────
Task            Potemkin Rate        Accuracy        N (valid) 
──────────────────────────────────────────────────────────────────────
Classify          97.0% ±  6.2%       51.5%        264
Generate           1.3% ±  1.3%       98.7%        77
Edit               4.9% ±  1.7%       95.1%        163
──────────────────────────────────────────────────────────────────────

Domain Breakdown (Classify):
  Game Theory               PR=  95.6%  Acc= 52.2%  N=90
  Literary Techniques       PR=  94.0%  Acc= 53.0%  N=100
  Psychology                PR= 102.7%  Acc= 48.6%  N=74

Results saved to /home/tpi/Documents/github_saram/PotemkinBenchmark_Reproducibility/

In [11]:
# =============================================================================
# COMPARISON TO ORIGINAL PAPER (Table 1)
# =============================================================================
# The original paper reported these overall Potemkin rates:
#   Classify: 55% ± 2%  (scaled by 2, since chance = 50%)
#   Generate: 40% ± 3%
#   Edit:     40% ± 2%
#
# Key comparison notes:
# 1. Different models: We use local 7B-8B models; original used 70B+ frontier models
# 2. Different annotation: We use LLM-as-judge; original used Upwork expert annotators
# 3. Same questions and concepts: Prompts are identical
# =============================================================================

original_table1 = {
    'Classify': {'rate': 55.0, 'se': 2.0},
    'Generate': {'rate': 40.0, 'se': 3.0},
    'Edit':     {'rate': 40.0, 'se': 2.0},
}

print(f'\n{"=" * 70}')
print(f'  COMPARISON: Our Replication vs. Original Paper (Table 1)')
print(f'{"=" * 70}')
print(f'{"Task":<12} {"Original":<18} {"Ours":<18} {"Diff":<12} {"Note"}')
print(f'{"─" * 70}')

for row in summary_rows:
    task = row['task']
    orig = original_table1.get(task, {})
    orig_rate = orig.get('rate', None)
    our_rate = row['potemkin_rate']

    if orig_rate is not None:
        diff = our_rate - orig_rate
        orig_str = f'{orig_rate:.1f}% ± {orig.get("se", 0):.1f}%'
        ours_str = f'{our_rate:.1f}% ± {row["se"]:.1f}%'
        diff_str = f'{diff:+.1f}%'
        note = 'Higher PR = worse understanding' if diff > 0 else 'Lower PR = better understanding'
        print(f'{task:<12} {orig_str:<18} {ours_str:<18} {diff_str:<12} {note}')

print(f'{"─" * 70}')
print()
print('Interpretation notes:')
print('- A HIGHER Potemkin rate means the model fails more often on tasks')
print('  for concepts it can define — i.e., more \"Potemkin understanding\".')
print('- Differences may arise from: (a) smaller model size, (b) LLM-as-judge')
print('  vs. human annotation, (c) stochastic variation.')
print('- The LLM-as-judge deviation is itself a reproducibility finding.')


  COMPARISON: Our Replication vs. Original Paper (Table 1)
Task         Original           Ours               Diff         Note
──────────────────────────────────────────────────────────────────────
Classify     55.0% ± 2.0%       97.0% ± 6.2%       +42.0%       Higher PR = worse understanding
Generate     40.0% ± 3.0%       1.3% ± 1.3%        -38.7%       Lower PR = better understanding
Edit         40.0% ± 2.0%       4.9% ± 1.7%        -35.1%       Lower PR = better understanding
──────────────────────────────────────────────────────────────────────

Interpretation notes:
- A HIGHER Potemkin rate means the model fails more often on tasks
  for concepts it can define — i.e., more "Potemkin understanding".
- Differences may arise from: (a) smaller model size, (b) LLM-as-judge
  vs. human annotation, (c) stochastic variation.
- The LLM-as-judge deviation is itself a reproducibility finding.


In [12]:
# =============================================================================
# SAVE EXPERIMENT METADATA (for reproducibility)
# =============================================================================

metadata = {
    'timestamp': datetime.now().isoformat(),
    'model_under_test': MODEL_UNDER_TEST,
    'judge_model': JUDGE_MODEL,
    'seed': SEED,
    'max_questions_per_task': MAX_QUESTIONS_PER_TASK,
    'n_define': len(df_define),
    'n_keystone_pass': len(keystone_concepts),
    'n_classify': len(df_classify),
    'n_generate': len(df_generate),
    'n_edit': len(df_edit),
    'define_accuracy': float(df_define['correct'].mean()),
    'results': summary_rows,
}

with open(os.path.join(OUTPUT_DIR, f'experiment_metadata_{model_short}.json'), 'w') as f:
    json.dump(metadata, f, indent=2)

print('Experiment metadata saved.')
print(f'\nAll outputs in: {OUTPUT_DIR}/')
print('Files:')
for f in sorted(os.listdir(OUTPUT_DIR)):
    fpath = os.path.join(OUTPUT_DIR, f)
    size = os.path.getsize(fpath)
    print(f'  {f} ({size:,} bytes)')

Experiment metadata saved.

All outputs in: /home/tpi/Documents/github_saram/PotemkinBenchmark_Reproducibility/Procedure1_Replication/Procedure1_Replication_Results/mistralai_Mistral-7B-Instruct-v0.3/
Files:
  classify_results_Mistral-7B-Instruct-v0.3.csv (68,974 bytes)
  define_results_Mistral-7B-Instruct-v0.3.csv (11,646 bytes)
  edit_results_Mistral-7B-Instruct-v0.3.csv (46,495 bytes)
  experiment_metadata_Mistral-7B-Instruct-v0.3.json (1,019 bytes)
  generate_results_Mistral-7B-Instruct-v0.3.csv (25,236 bytes)
  procedure1_summary_Mistral-7B-Instruct-v0.3.csv (282 bytes)


In [ ]:
# # =============================================================================
# # MULTI-MODEL RUNNER
# # =============================================================================
# # Runs all models in MODELS_TO_RUN sequentially.
# # Run cells 1–6 (config, imports, prompts, helpers, data loading) FIRST,
# # then execute this single cell to evaluate all three models back-to-back.
# #
# # Between each model the GPU model cache is cleared so the next model can
# # load within the 48 GB VRAM budget.
# # =============================================================================

# import gc
# import torch
# from utils import _model_cache


# def _clear_model_cache():
#     """Unload all cached models to free GPU VRAM before loading the next model."""
#     _model_cache.clear()
#     gc.collect()
#     if torch.cuda.is_available():
#         torch.cuda.empty_cache()
#     print("  GPU cache cleared.\n")


# def run_experiment_for_model(model_id: str, judge_id: str) -> list:
#     """
#     Run all four Procedure 1 tasks (Define, Classify, Generate, Edit) for one
#     model and return a list of summary-row dicts (one per task).
#     """
#     model_short = model_id.split('/')[-1]
#     out_dir = os.path.join(
#         PROJECT_ROOT, 'Procedure1_Replication_Results',
#         model_id.replace('/', '_')
#     )
#     os.makedirs(out_dir, exist_ok=True)

#     ts = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
#     print(f"\n{'='*70}")
#     print(f"  MODEL : {model_id}")
#     print(f"  JUDGE : {judge_id}")
#     print(f"  START : {ts}")
#     print(f"  OUTPUT: {out_dir}")
#     print(f"{'='*70}\n")

#     # ------------------------------------------------------------------ DEFINE
#     print("--- Task 1: Define ---")
#     _define_results = []
#     for item in tqdm(define_questions, desc='Define', leave=False):
#         concept = item['Concept']
#         domain  = item.get('Domain', get_domain(concept))
#         prompt  = DEFINE_PROMPT_TEMPLATE.format(concept=concept)
#         full_response = safe_inference(prompt, model_id)
#         definition    = extract_final_answer(full_response) or full_response.strip()
#         print(definition[:200])  # Print first 200 chars of definition for debugging
#         judge_prompt = DEFINE_JUDGE_PROMPT_TEMPLATE.format(
#             concept=concept, definition=definition
#         )
#         verdict    = judge_output(judge_prompt, judge_id)
#         print(f"  Verdict: {verdict}")
#         is_correct = (verdict == 1)
#         _define_results.append({
#             'concept': concept, 'domain': domain, 'model': model_short,
#             'definition': definition, 'judge_verdict': verdict, 'correct': is_correct,
#         })

#     df_def = pd.DataFrame(_define_results)
#     df_def.to_csv(os.path.join(out_dir, f'define_results_{model_short}.csv'), index=False)
#     _keystone = set(df_def[df_def['correct']]['concept'].tolist())
#     print(f"  Define: {df_def['correct'].sum()}/{len(df_def)} correct  |  "
#           f"Keystone set: {len(_keystone)} concepts")

#     # --------------------------------------------------------------- CLASSIFY
#     print("--- Task 2: Classify ---")
#     _cls_data    = [q for q in classify_data if q['concept'] in _keystone]
#     _cls_sampled = subsample(_cls_data, MAX_QUESTIONS_PER_TASK)
#     _classify_results = []

#     for item in tqdm(_cls_sampled, desc='Classify', leave=False):
#         concept    = item['concept']
#         prompt     = CLASSIFY_ANSWER_PROMPT_TEMPLATE.format(question=item['prompt'])
#         true_label = item['true_label']
#         full_response = safe_inference(prompt, model_id)
#         answer        = extract_final_answer(full_response) or full_response

#         ans_lower = answer.strip().lower()
#         if 'yes' in ans_lower[:20]:
#             model_label = 'yes'
#         elif 'no' in ans_lower[:20]:
#             model_label = 'no'
#         else:
#             m = re.search(r'ANSWER:\s*(Yes|No)', full_response, re.I)
#             model_label = m.group(1).lower() if m else 'unparseable'

#         _classify_results.append({
#             'concept': concept, 'domain': item['domain'], 'model': model_short,
#             'model_label': model_label, 'true_label': true_label,
#             'correct': (model_label == true_label), 'raw_answer': answer[:200],
#         })

#     df_cls = pd.DataFrame(_classify_results)
#     df_cls.to_csv(os.path.join(out_dir, f'classify_results_{model_short}.csv'), index=False)
#     valid_cls = df_cls[df_cls['model_label'] != 'unparseable']
#     acc_cls   = valid_cls['correct'].mean() if len(valid_cls) else 0
#     pr_cls    = (1 - acc_cls) * 2 * 100
#     print(f"  Classify: acc={acc_cls*100:.1f}%  potemkin_rate={pr_cls:.1f}%  "
#           f"valid={len(valid_cls)}/{len(df_cls)}")

#     # --------------------------------------------------------------- GENERATE
#     print("--- Task 3: Generate ---")
#     _gen_data    = [q for q in generate_data if q['concept'] in _keystone]
#     _gen_sampled = subsample(_gen_data, MAX_QUESTIONS_PER_TASK)
#     _generate_results = []

#     for item in tqdm(_gen_sampled, desc='Generate', leave=False):
#         concept    = item['concept']
#         task_prompt = item['prompt']
#         prompt      = GENERATE_ANSWER_PROMPT_TEMPLATE.format(question=task_prompt)
#         full_response = safe_inference(prompt, model_id)
#         output        = extract_final_answer(full_response) or full_response.strip()

#         judge_prompt = GENERATE_JUDGE_PROMPT_TEMPLATE.format(
#             concept=concept, prompt=task_prompt, output=output
#         )
#         verdict    = judge_output(judge_prompt, judge_id)
#         _generate_results.append({
#             'concept': concept, 'domain': item['domain'], 'model': model_short,
#             'output': output[:500], 'judge_verdict': verdict, 'correct': (verdict == 1),
#         })

#     df_gen = pd.DataFrame(_generate_results)
#     df_gen.to_csv(os.path.join(out_dir, f'generate_results_{model_short}.csv'), index=False)
#     acc_gen = df_gen['correct'].mean() if len(df_gen) else 0
#     pr_gen  = (1 - acc_gen) * 100
#     print(f"  Generate: acc={acc_gen*100:.1f}%  potemkin_rate={pr_gen:.1f}%  N={len(df_gen)}")

#     # ------------------------------------------------------------------- EDIT
#     print("--- Task 4: Edit ---")
#     _edit_data    = [q for q in edit_data if q['concept'] in _keystone]
#     _edit_sampled = subsample(_edit_data, MAX_QUESTIONS_PER_TASK)
#     _edit_results = []

#     for item in tqdm(_edit_sampled, desc='Edit', leave=False):
#         concept    = item['concept']
#         task_prompt = item['prompt']
#         prompt      = EDIT_ANSWER_PROMPT_TEMPLATE.format(question=task_prompt)
#         full_response = safe_inference(prompt, model_id)
#         output        = extract_final_answer(full_response) or full_response.strip()

#         judge_prompt = EDIT_JUDGE_PROMPT_TEMPLATE.format(
#             concept=concept, prompt=task_prompt, output=output
#         )
#         verdict    = judge_output(judge_prompt, judge_id)
#         _edit_results.append({
#             'concept': concept, 'domain': item['domain'], 'model': model_short,
#             'output': output[:500], 'judge_verdict': verdict, 'correct': (verdict == 1),
#         })

#     df_edit = pd.DataFrame(_edit_results)
#     df_edit.to_csv(os.path.join(out_dir, f'edit_results_{model_short}.csv'), index=False)
#     acc_edit = df_edit['correct'].mean() if len(df_edit) else 0
#     pr_edit  = (1 - acc_edit) * 100
#     print(f"  Edit:     acc={acc_edit*100:.1f}%  potemkin_rate={pr_edit:.1f}%  N={len(df_edit)}")

#     # --------------------------------------------------------- PER-MODEL SUMMARY
#     summary_rows = []
#     for task_name, df_t, scale2 in [
#         ('Classify', df_cls, True), ('Generate', df_gen, False), ('Edit', df_edit, False)
#     ]:
#         valid = df_t[df_t['model_label'] != 'unparseable'] if task_name == 'Classify' else df_t
#         if len(valid) == 0:
#             continue
#         acc = valid['correct'].mean()
#         se  = math.sqrt(acc * (1 - acc) / max(len(valid), 1))
#         mult = 2 if scale2 else 1
#         summary_rows.append({
#             'model': model_short, 'model_full': model_id,
#             'judge': judge_id.split('/')[-1],
#             'task': task_name,
#             'potemkin_rate': round((1 - acc) * mult * 100, 2),
#             'se': round(se * mult * 100, 2),
#             'accuracy': round(acc * 100, 2),
#             'n': len(valid),
#             'timestamp': ts,
#         })

#     pd.DataFrame(summary_rows).to_csv(
#         os.path.join(out_dir, f'procedure1_summary_{model_short}.csv'), index=False
#     )
#     metadata = {
#         'timestamp': ts, 'model_under_test': model_id, 'judge_model': judge_id,
#         'seed': SEED, 'max_questions_per_task': MAX_QUESTIONS_PER_TASK,
#         'n_define': len(df_def), 'n_keystone_pass': len(_keystone),
#         'n_classify': len(df_cls), 'n_generate': len(df_gen), 'n_edit': len(df_edit),
#         'define_accuracy': float(df_def['correct'].mean()), 'results': summary_rows,
#     }
#     with open(os.path.join(out_dir, f'experiment_metadata_{model_short}.json'), 'w') as fh:
#         json.dump(metadata, fh, indent=2)

#     print(f"\n  Saved results to {out_dir}/")
#     return summary_rows


# # =============================================================================
# # LOOP — run each model, clear VRAM between runs
# # =============================================================================

# all_summary_rows = []

# for _model_id in MODELS_TO_RUN:
#     _rows = run_experiment_for_model(_model_id, judge_id=_model_id)
#     all_summary_rows.extend(_rows)
#     _clear_model_cache()

# # =============================================================================
# # CROSS-MODEL COMPARISON TABLE
# # =============================================================================

# original_table1 = {
#     'Classify': {'rate': 55.0, 'se': 2.0},
#     'Generate': {'rate': 40.0, 'se': 3.0},
#     'Edit':     {'rate': 40.0, 'se': 2.0},
# }

# print(f"\n{'='*90}")
# print(f"  CROSS-MODEL COMPARISON — Procedure 1 Potemkin Rates")
# print(f"{'='*90}")
# print(f"{'Model':<45} {'Task':<12} {'Our PR':<14} {'Original PR':<14} {'Diff'}")
# print(f"{'─'*90}")

# df_all = pd.DataFrame(all_summary_rows)
# for _, row in df_all.iterrows():
#     orig    = original_table1.get(row['task'], {})
#     orig_pr = orig.get('rate', float('nan'))
#     diff    = row['potemkin_rate'] - orig_pr if not math.isnan(orig_pr) else float('nan')
#     diff_s  = f'{diff:+.1f}%' if not math.isnan(diff) else 'N/A'
#     print(f"{row['model']:<45} {row['task']:<12} "
#           f"{row['potemkin_rate']:>5.1f}% ±{row['se']:>4.1f}%   "
#           f"{orig_pr:>5.1f}% ±{orig.get('se',0):>3.1f}%   {diff_s}")

# print(f"{'─'*90}")

# # Save combined CSV
# combined_path = os.path.join(PROJECT_ROOT, 'Procedure1_Replication_Results', 'all_models_summary.csv')
# df_all.to_csv(combined_path, index=False)
# print(f"\nCombined results saved to: {combined_path}")

In [ ]:
# pip install --upgrade "transformers==4.51.3" "accelerate>=1.2.1"
# !pip install "transformers==4.51.3" "accelerate>=1.2.1"